In [1]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import duckdb
from pathlib import Path
from typing import List
import os

In [3]:
# Store database at project root
DB_NAME = Path("../amazing.duckdb") 
# Go up one level from current directory to get to project root
data_folder = Path("../data") 
# For absolute certainty, you could use the absolute path
# data_folder = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/data")

con = duckdb.connect(str(DB_NAME))

In [4]:
def get_file_paths() -> List[str]:
    return sorted([str(p.resolve()) for p in data_folder.glob("*.csv")])

def load_new_files(con, file_paths: List[str]):
    print(f"Début du chargement de {len(file_paths)} fichier(s)...\n")
    for i, path in enumerate(file_paths, 1):
        filename = os.path.basename(path)

        print(f"[{i}/{len(file_paths)}] Vérification de {filename}...")
        already_loaded = con.execute(
            "SELECT 1 FROM loaded_files WHERE filename = ?", [filename]
        ).fetchone()

        if already_loaded:
            print(f"{filename} déjà chargé. Ignoré.\n")
            continue

        print(f"⬆Chargement de {filename} dans all_events...")
        con.execute(f"""
            INSERT INTO all_events
            SELECT * FROM read_csv_auto('{path}', AUTO_DETECT=TRUE, SAMPLE_SIZE=-1)
        """)
        con.execute("INSERT INTO loaded_files VALUES (?)", [filename])
        print(f"{filename} ajouté avec succès à la base.\n")

    print("Chargement terminé.\n")

def init_loaded_table(con):
    print("Initialisation de la table 'loaded_files'...")
    con.execute("""
        CREATE TABLE IF NOT EXISTS loaded_files (
            filename TEXT PRIMARY KEY
        );
    """)
    print("Table 'loaded_files' prête.\n")

def create_all_events_table(con):
    con.execute("""
        CREATE TABLE IF NOT EXISTS all_events (
            event_time TIMESTAMP,
            event_type TEXT,
            product_id TEXT,
            category_id TEXT,
            category_code TEXT,
            brand TEXT,
            price DOUBLE,
            user_id TEXT,
            user_session TEXT
        );
    """)

In [5]:


# Initialisation des tables
init_loaded_table(con)
create_all_events_table(con)

# Chargement des fichiers
files = get_file_paths()
load_new_files(con, files)

# Test génération de la table events_tables
create_all_events_table(con)



Initialisation de la table 'loaded_files'...
Table 'loaded_files' prête.

Début du chargement de 1 fichier(s)...

[1/1] Vérification de 2020-Feb.csv...


⬆Chargement de 2020-Feb.csv dans all_events...
2020-Feb.csv ajouté avec succès à la base.

Chargement terminé.



In [6]:
nb_users = con.execute("SELECT COUNT(*) FROM all_events").fetchone()[0]

print(f"Taille de la table all_events : {nb_users} logs")

df_purchase = con.execute("SELECT * FROM all_events WHERE user_id = '535135317' LIMIT 10").fetch_df()
print(df_purchase)


Taille de la table all_events : 55318565 logs
           event_time event_type product_id          category_id  \
0 2020-02-01 20:22:57       view   21404116  2232732082063278200   
1 2020-02-03 23:37:15       view   26500142  2053013553140465927   
2 2020-02-03 23:37:21       cart   26500142  2053013553140465927   
3 2020-02-03 23:37:30   purchase   26500142  2053013553140465927   
4 2020-02-03 23:38:18       view   26500142  2053013553140465927   
5 2020-02-03 23:38:31       view    5500321  2232732093941547400   
6 2020-02-03 23:39:21       view    5500321  2232732093941547400   
7 2020-02-04 19:45:03       view   15100009  2232732107413652135   
8 2020-02-04 19:45:48       view   15700096  2232732094134485388   
9 2020-02-04 19:46:58       view   15700107  2232732094134485388   

               category_code    brand   price    user_id  \
0         electronics.clocks    casio   37.58  535135317   
1                  kids.toys  lucente  228.83  535135317   
2                  kids.t

In [8]:
# Fermeture de la connexion DuckDB
con.close()